# Running Power Flow In The Loop with Unit Commitment

> **Set up**
>
> To run this notebook, first install the Julia kernel for Jupyter Notebooks using [IJulia](https://julialang.github.io/IJulia.jl/stable/manual/installation/), then [create an environment](https://pkgdocs.julialang.org/v1/environments/) for this tutorial with the packages listed with `using <PackageName>` further down.
>
> This tutorial has demonstrated compatibility with these package versions. If you run into any errors, first check your package versions for consistency using `Pkg.status()`.
>
 > ```
 > Status `~/work/PowerSimulations.jl/PowerSimulations.jl/docs/Project.toml`
 >   [336ed68f] CSV v0.10.16
 >   [a93c6f00] DataFrames v1.8.2
 >   [864edb3b] DataStructures v0.19.5
 >   [e30172f5] Documenter v1.17.0
 >   [d12716ef] DocumenterInterLinks v1.1.0
 >   [35a29f4d] DocumenterTools v0.1.21
 >   [87dc4568] HiGHS v1.23.0
 >   [fc1677e0] HydroPowerSimulations v0.17.0
 >   [2cd47ed4] InfrastructureSystems v3.6.0
 >   [4076af6c] JuMP v1.30.1
 >   [23fbe1c1] Latexify v0.16.10
 >   [98b081ad] Literate v2.21.0
 >   [94fada2c] PowerFlows v0.22.0
 >   [bed98974] PowerNetworkMatrices v0.24.2
 >   [e690365d] PowerSimulations v0.36.3 `~/work/PowerSimulations.jl/PowerSimulations.jl`
 >   [f00506e0] PowerSystemCaseBuilder v2.3.0
 >   [bcd98974] PowerSystems v5.11.1
 >   [08abe8d2] PrettyTables v3.3.2
 > ⌅ [9e3dc215] TimeSeries v0.25.2
 > Info Packages marked with ⌅ have new versions available but compatibility constraints restrict them from upgrading. To see why use `status --outdated`
 > 
 > ```


In this tutorial, you'll configure a unit commitment (UC) simulation that
automatically calls an AC power flow solver from
[`PowerFlows.jl`](https://sienna-platform.github.io/PowerFlows.jl/stable/) at every dispatch interval.

You'll validate the committed dispatch against the full AC network model at each hour, check for branch overloads, and export the results to PSS/e format if needed for further analysis.

This tutorial builds on the tutorial for
Running a Multi-Stage Production Cost Simulation.

## Setup

Load the needed packages and define basic inputs:

In [ ]:
using PowerSystemCaseBuilder
using PowerSimulations
using HydroPowerSimulations
using PowerFlows
using PowerSystems
using DataFrames
using HiGHS
using Dates
using Logging
const SCENARIO_NAME = "uc_with_pf"
const INLOOP_SYSTEM = "modified_RTS_GMLC_DA_sys"
const INLOOP_SIM_STEPS = 1

run_dir = joinpath(".", "uc_power_flow_in_the_loop_results")
mkpath(run_dir)
export_dir = joinpath(run_dir, "psse_exports")
mkpath(export_dir)

Build a test `PowerSystems.System` via
`PowerSystemCaseBuilder.build_system`.
We use `modified_RTS_GMLC_DA_sys` (24+ DA forecast steps):

In [ ]:
sys = build_system(
    PSISystems,
    INLOOP_SYSTEM;
    skip_serialization = true,
    runchecks = false,
)

## Configuring the Power Flow Solver

Create an `PowerFlows.ACPowerFlow` solver. We attach a
`PowerFlows.PSSEExportPowerFlow` exporter so that
we can automatically write a PSS/e `.raw` file for each solved interval.

In [ ]:
psse_export = PSSEExportPowerFlow(;
    psse_version = :v33,
    export_dir = export_dir,
    overwrite = true,
)

power_flow_model = ACPowerFlow(; exporter = psse_export)

### Alternative: Fast Decoupled (FDNR) solver

PowerFlows 0.22 adds a fast-decoupled AC solver, selected via the solver type parameter of
`PowerFlows.ACPowerFlow`. It plugs into `power_flow_evaluation` exactly like the
default Newton-Raphson solver (same auxiliary variables and PSS/e exports) and matches its
converged solution, but is often faster on large networks. Optionally hand off to an exact
Newton solver for the final refinement:

```julia
using PowerFlows: FastDecoupledACPowerFlow, NewtonRaphsonACPowerFlow

fd_power_flow_model = ACPowerFlow{FastDecoupledACPowerFlow}(;
    exporter = psse_export,
    solver_settings = Dict{Symbol, Any}(
        :handoff_solver => NewtonRaphsonACPowerFlow,  # refine FD result with Newton-Raphson
        :handoff_tol => 1e-3,                         # FD-stage exit tolerance before handoff
    ),
)
```

Use it in place of `power_flow_model` below. Omit `solver_settings` for pure fast decoupled, or
use the `FastDecoupledFixed` alias for the formulation-agnostic frozen-Jacobian variant.

## Building the UC Problem Template

Create a `ProblemTemplate` with a `NetworkModel` that uses
`PTDFPowerModel` and `power_flow_evaluation` set to the solver we just configured.
Assign device formulations with `set_device_model!`. This is the key step that enables power flow in the loop:

In [ ]:
template_uc = ProblemTemplate(
    NetworkModel(
        PTDFPowerModel;
        use_slacks = true,
        power_flow_evaluation = power_flow_model,
    ),
)

set_device_model!(
    template_uc,
    ThermalStandard,
    ThermalStandardUnitCommitment,
)
set_device_model!(template_uc, RenewableDispatch, RenewableFullDispatch)
set_device_model!(template_uc, RenewableNonDispatch, FixedOutput)
set_device_model!(template_uc, PowerLoad, StaticPowerLoad)
set_device_model!(template_uc, HydroDispatch, HydroDispatchRunOfRiver)
set_device_model!(template_uc, Line, StaticBranchUnbounded)
set_device_model!(template_uc, Transformer2W, StaticBranchUnbounded)
set_device_model!(template_uc, MonitoredLine, StaticBranch)

`use_slacks = true` allows the simulation to remain feasible when there is a small
mismatch between the PTDF-based UC network model and the full AC power flow.

## Building and Executing the Simulation

Follow the same pattern as in the tutorial for Running a Multi-Stage Production Cost Simulation: package the UC `DecisionModel` in `SimulationModels`, define a
`SimulationSequence` with `InterProblemChronology`, construct a
`Simulation`, then `build!` and `execute!` it for one simulation step.
The in-step horizon on `modified_RTS_GMLC_DA_sys` provides 24 hourly realized rows.

In [ ]:
solver = optimizer_with_attributes(
    HiGHS.Optimizer,
    "log_to_console" => false,
    "mip_rel_gap" => 0.05,
    "time_limit" => 900.0,
)

models = SimulationModels(;
    decision_models = [
        DecisionModel(
            template_uc,
            sys;
            name = "UC",
            optimizer = solver,
            store_variable_names = true,
        ),
    ],
)

sequence = SimulationSequence(;
    models = models,
    ini_cond_chronology = InterProblemChronology(),
)

sim = Simulation(;
    name = SCENARIO_NAME,
    steps = INLOOP_SIM_STEPS,
    models = models,
    sequence = sequence,
    simulation_folder = run_dir,
)

build!(sim; console_level = Logging.Error, file_level = Logging.Error)
execute!(sim; enable_progress_bar = true)

## Loading Simulation Results

Load the simulation results with `SimulationResults`
and extract the UC problem results with `get_decision_problem_results`:

In [ ]:
sim_results = SimulationResults(sim)
uc_results = get_decision_problem_results(sim_results, "UC")

Notice the "UC Problem Auxiliary variables Results" table, which lists the active and
reactive power flow and bus voltage magnitude and angle results from the AC power flow  (e.g., `PowerFlowBranchReactivePowerFromTo__Line`, `PowerFlowVoltageMagnitude__ACBus`).
These are not output when a UC problem is run alone.

## PTDF UC Flows vs. AC Power Flow In the Loop

Now, we'll compare the PTDF UC flows to the AC power flow results.

The UC stage optimizes flows using the PTDF network model, and the line flows are not variables; they are recorded in the `PTDFBranchFlow__*` expressions.

First, load in the PTDF branch flows for one line with `read_realized_expression`:

In [ ]:
ptdf_flows = read_realized_expression(uc_results, "PTDFBranchFlow__Line")
example_line = first(unique(ptdf_flows.name))
ptdf_line = filter(row -> row.name == example_line, ptdf_flows)

After each in-step hour, the AC power flow writes branch flows into auxiliary variables
such as `PowerFlowBranchActivePowerFromTo__Line`.
Load those flows with `read_realized_aux_variable` and compare PTDF expression flows to AC flows for our selected line:

In [ ]:
pf_flows_ft = read_realized_aux_variable(
    uc_results,
    "PowerFlowBranchActivePowerFromTo__Line",
)
pf_line = filter(row -> row.name == example_line, pf_flows_ft)

innerjoin(
    rename(select(ptdf_line, :DateTime, :value), :value => :PTDF_MW),
    rename(select(pf_line, :DateTime, :value), :value => :AC_PF_MW);
    on = :DateTime,
)

With `use_slacks = true` in the template, small differences between PTDF and AC flows
are expected — the UC model can absorb minor mismatches. The AC auxiliary flows should track
the PTDF values closely when the network is well conditioned.

## Checking for Branch Overloads

Now, we will run a post-hoc check on our UC results to scan every interval for branches whose AC flow exceeds the thermal rating.

First, build a helper to extract branch flow limits from the `PowerSystems.System` — `name` in the
auxiliary results matches the branch name on each `PowerSystems.ACBranch`.
Limits are scaled with `PowerSystems.get_base_power`. AC lines and transformers use
`PowerSystems.get_rating`; HVDC branches use active-power limits instead.

In [ ]:
function _branch_flow_limit_mw(branch, sys)
    base_power = get_base_power(sys)
    try
        if branch isa TwoTerminalVSCLine
            return get_rating(branch) * base_power
        elseif branch isa TwoTerminalHVDC
            return max(
                get_active_power_limits_from(branch).max,
                get_active_power_limits_to(branch).max,
            )
        elseif branch isa GenericArcImpedance
            return get_max_flow(branch) * base_power
        else
            return get_rating(branch) * base_power
        end
    catch e
        e isa MethodError || rethrow(e)
        @warn "Could not get rating for $(typeof(branch)): $e — treating as unlimited"
        return Inf
    end
end

Next, build a lookup for each `PowerSystems.ACBranch` keyed by branch name using `PowerSystems.get_components` and
`get_name`:

In [ ]:
ratings = Dict{String, Float64}()
for b in get_components(ACBranch, sys)
    ratings[get_name(b)] = _branch_flow_limit_mw(b, sys)
end

Then, scan every interval for branches whose AC flow exceeds the thermal rating:

In [ ]:
overloads = DataFrame(;
    DateTime = Dates.DateTime[],
    name = String[],
    P_from_to = Float64[],
    rating = Float64[],
)

for row in eachrow(pf_flows_ft)
    rating = get(ratings, row.name, Inf)
    if abs(row.value) > rating
        push!(overloads, (row.DateTime, row.name, row.value, rating))
    end
end

overloads

Notice we have multiple overloads in this example.
Each row identifies a congestion event: the interval (`DateTime`),
branch name (`name`), actual AC flow in MW (`P_from_to`), and thermal rating (`rating`).
In-loop AC power flow records realized flows but does not add thermal-limit constraints
to the UC optimization.
The UC model's network representation could be too loose if overloads appear — add transmission
constraints, re-run UC, and repeat the AC power flow check until the table is empty.

## PSS/e Export Files

The `PowerFlows.PSSEExportPowerFlow` exporter writes under
`export_dir` (here, `results/psse_exports`). After running, we have one folder per solved
interval in the 24-hour in-step horizon, plus the 24-hour lookahead:

In [ ]:
psse_export_folders = readdir(export_dir)
length(psse_export_folders)

Each folder contains the PSS/e `.raw` file for that interval, available for further analysis:

In [ ]:
first_subdir = first(sort(psse_export_folders))
readdir(joinpath(export_dir, first_subdir))

## Next Steps

- Tighten the UC network model with transmission constraints if needed until overloads are removed — see
  the formulation library Network and `PowerSystems.Branch` Formulations pages.
- See the Solving a Power Flow
  tutorial in PowerFlows.jl for standalone power flow examples.

In [ ]:
rm(run_dir; force = true, recursive = true) #hide